In [1]:
import math
import numpy as np 
import pandas as pd
from collections import defaultdict

In [2]:
class XGBoostModel():

    def __init__(self, params,random_seed=None):
        self.params = defaultdict(params)
        self.subsamples = self.params['subsamples'] \
            if self.params['subsamples'] else 0.1
        self.learning_rate = self.params['learning_rate'] \
            if self.params['learning_rate'] else 0.3
        self.base_prediction = self.params['base_score'] \
            if self.params['base_score'] else 0.5
        self.max_depth = self.params['max_depth'] \
            if self.params['max_depth'] else 5

        self.rng = np.random.default_rng(seed=random_seed)
        

In contrast to boosting in the classic GBM, instead of computing residuals between the current predictions and the target, we compute gradients and hessians of the loss function with respect to the current predictions, and instead of predicting residuals with a decision tree, we fit a special XGBoost tree booster, using the gradients and hessians

In [ ]:
def fit(self,X, y, objective, n_estimators):
    current_prediction = self.base_prediction*np.ones(shape=y.shape) #Starting mein model will produce same prediction. Then we will generate gradient and hessian and the work on it.
    self.models = []
    sample_idx = 0

    for i in range(n_estimators):
        gradients = objective.gradient(y,current_prediction) #current prediction, actual se kitna glt hain.
        hessians = objective.hessian(y, current_prediction) #rate change of the loss function

        if self.subsample == 1:
            sample_idx= None
        else: 
            sample_idx = self.rng.choice(len(y), size=math.floor(self.subsample*len(y)), replace = False)   #subsample*len(y) means total y me se kitne samples select krne hain. 
            #self.rng.choice-> randomly select the samples 

    Tree  = TreeBooster( X, gradients, hessians, self.params, self.max_depth, sample_idx) #create new trees

    current_prediction += self.learning_rate *Tree.predict(X)  #update the predictions 

    self.models.append(Tree)

def predict(self, X): 
    return (
        self.base_prediction + self.learning_rate * np.sum([tree.predict(X) for tree in self.Tree], axis = 0)
    )

XGBoostModel.fit = fit
XGBoostModel.predict = predict 

$$
\text{Base Prediction} = \text{Base Prediction} + \eta \sum_{k=1}^{K} \text{Tree}_k(X)
$$

Now we recursively build a binary tree structure by finding the best split rule for each node in the tree. The main difference is the criterion for evaluating splits and the way that we define a leaf's predicted value. Instead of being functions of the target values of the instances in each node, the criterion and predicted values are functions of the instance gradients and hessians

In [ ]:
class TreeBooster():
    def __init__ (self, X, g, h, params, max_depth, idx = None):

        self.params = params
        self.max_depth = max_depth 
        self.min_child_weight = params['min_child_weight'] \
            if params['min_child_weight'] else 1.0 #Ye basically check karta hai ki child node mein enough Hessian weight hai ya nahi.
        self.reg_lambda = params['reg_lambda'] \
            if params['reg_lambda'] else 1.0 # for L2 regularization, lambda. Controls the overfitting
        self.gamma = params['gamma'] \
        if params['gamma'] else 0.0 #mini improvement required to initiate split 
        self.colsample_bynode = params['colsample_bynode'] \
        if params['colsample_bynode'] else 1.0 # Har node par kitne features consider karne hain.
        

        if isinstance(g, pd.Series):
            g = g.values  # Agar g aur h pandas Series hain, toh convert them into numpy array
        if isinstance(h, pd.Series):
            h = h.values

        if idx is None :  #Agar koi idxs nahi diya gaya, iska matlab ye root node hai.
            idx = np.arrange(len(g)) # if len(g) == 5, idx = [0,1,2,3,4]. Root node contains all the samples

        self.X, self.h, self.g, self.idx = X, h,g,idx

        self.n = len(idx) #current node me kitne samples hain 
        self.c = X.shape[1] #total number of features/columns

        # if, X.shape = (100, 5); 100 samples, 5 features

        #Calculating optimal weight of the leaf 
        self.value = -g[idx].sum() / (h[idx] + self.reg_lambda)

        self.best_score = 0 #it will track if we found any best score or not

        if self.best_score> 0:  #agar accha split mil gaya toh tree ko grow karo
            self.insert_child_node()

    #splitting logic 

    def insert_child_node(self):

        for i in range(self.c):
            self.find_better_split(i) 

        if self.is_leaf: #agar best split nahi mila = leaf node reached
            return 

        x = self.X.values[self.idx,self.split_feature_idx] #us index se lekar us feature tak ke index tak data divide krna. 
        left_leaf = np.nonzero(x<self.threshold)[0] #np.nonzero returns a tupple. for ex (array([0, 1]),).. [0] make sure that we have [0,1] in our variable
        right_leaf = np.nonzero(x>= self.threshold)[0]

        self.left= TreeBooster(self.X, self.g, self.h, self.params, self.max_depth-1, self.idx[left_leaf])
        self.right= TreeBooster(self.X, self.g, self.h, self.params, self.max_depth-1, self.idx[right_leaf])
        #             Root 
        #           /      \
        #       Left        Right
        #      /    \       /    \
        #    LL     LR     RL     RR

        #(g,h,idx,value for4 every node) 


    def is_leaf(self):
        return self.best_score == 0

    #function which decides on which threshold and feature should we split our tree
    def find_better_split(self, feature_idx):
        x = self.X.values[self.idx, feature_idx] #self.idxs ensure karta hai ki sirf current node ke samples liye ja rahe hain.
        




self.value = -g[idx].sum() / (h[idx] + self.reg_lambda) is actually the formula below

$$
w^* = -\frac{\sum g_i}{\sum h_i + \lambda}
$$

g = gradient
h = hessian
λ = regularization
w* = leaf prediction/update

TreeBooster Flow

```text
X
y
 ↓
Current Prediction
 ↓
Gradient + Hessian
 ↓
TreeBooster(X, g, h, ...)
 ↓
┌────────────────────────────┐
│ Root Node                  │
│                            │
│ Calculate Leaf Value       │
│ Find Best Feature / Split  │
└──────────────┬─────────────┘
               ↓
          Best Split?
          /          \
        YES           NO
         ↓             ↓
   Split Data        Leaf
    /      \
   /        \
 Left      Right
  ↓          ↓
TreeBooster  TreeBooster
```
```


Splitting process
```text
Features
   ↓
Candidate Thresholds
   ↓
Gradient + Hessian
   ↓
Check `min_child_weight`
   ↓
Calculate Gain
   ↓
Apply `gamma` + Regularization
   ↓
Choose Best Split
   ↓
Check `max_depth`
   ↓
Split / Leaf

colsample_bynode → Which features?
threshold → Where to split?
gradient + hessian → How good is the split?
min_child_weight + gamma → Should we allow the split?
max_depth → How deep can we go?